# 1. Exploring the data

What one day (or one minute) of Binance BTCUSDT looks like through the simulator's eyes: the best bid and ask
over time, the spread, the trades, and whether the order book's imbalance says anything about the next move.

Everything here reads a packed `.bse` file with `blocksprint_sim.data_bbo` and `data_trades` (the book rebuilt
with the simulator's own rules). It runs on the committed one-minute slice; to explore the recorded day, set
`DATA`, `FROM` and `TO` in the next cell (the packed dataset isn't in git: `docs/data_pipeline.md`).

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import display

import blocksprint_sim as bs
from research.analytics import report

# The repository root, wherever the notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "analytics").is_dir())

DATA = ROOT / "tests/sim/data/btcusdt_2026-11-05_0200_1m.bse"
FROM, TO = None, None  # the whole file
# DATA = ROOT / "data/packed/binance_spot_btc_eth_1110/BTCUSDT.bse"
# FROM, TO = "2026-11-10T14:00:00", "2026-11-10T15:00:00"   # UTC


def ns(text):
    return 0 if text is None else int(np.datetime64(text, "ns").astype(np.int64))

In [ ]:
bbo = bs.data_bbo(str(DATA), ns(FROM), ns(TO))
trades = bs.data_trades(str(DATA), ns(FROM), ns(TO))
book = pl.DataFrame({
    "ts_ns": bbo["ts_ns"].astype(np.int64), "bid": bbo["bid"] / 1e8, "bid_qty": bbo["bid_qty"] / 1e8,
    "ask": bbo["ask"] / 1e8, "ask_qty": bbo["ask_qty"] / 1e8,
}).filter((pl.col("bid") > 0) & (pl.col("ask") > 0)).with_columns(((pl.col("bid") + pl.col("ask")) / 2).alias("mid"))
tape = pl.DataFrame({"ts_ns": trades["ts_ns"].astype(np.int64), "px": trades["px"] / 1e8,
                     "qty": trades["qty"] / 1e8, "buy": trades["side"] == 0})
tick = float(np.round(np.diff(np.unique(book["bid"].to_numpy())).min(), 8))
span_s = (book["ts_ns"][-1] - book["ts_ns"][0]) / 1e9
print(f"{book.height:,} changes of the best bid or ask and {tape.height:,} trades over {span_s:,.0f} s; tick {tick}")

## The spread

How much of the time the spread is 1, 2, 3 ... ticks wide, weighting each state by how long it lasted. BTCUSDT
sits at one tick almost all the time: a market maker there competes on queue position, not on price.

In [ ]:
dur = np.diff(np.append(book["ts_ns"].to_numpy(), book["ts_ns"][-1]))
ticks = np.round(((book["ask"] - book["bid"]) / tick).to_numpy()).astype(int)
labels = ["1", "2", "3", "4", "5+"]
share = [dur[np.minimum(ticks, 5) == k].sum() / dur.sum() for k in range(1, 6)]
display(report.frame(report.card("Share of time by spread (ticks)", report.bar_chart(
    labels, share, y_title="share of time", percent=True, digits=3, x_title="spread (ticks)")), 420))

## The mid and the trades

The mid over time, and the traded volume per bucket split by the aggressor's side (a buy aggressor lifted the
ask). Buckets: at least 30 over the window.

In [ ]:
every = report.auto_every(int(book["ts_ns"][-1] - book["ts_ns"][0]))
step = {"1s": 1, "5s": 5, "10s": 10, "30s": 30, "1m": 60, "5m": 300, "15m": 900, "1h": 3600}[every] * 1_000_000_000
t = book["ts_ns"].to_numpy()
axis = report._time_axis(int(t[0]), int(t[-1]))
mid_chart = report.line_chart(t, [("Mid", book["mid"].to_numpy())], x_labels=[report._utc(int(v)) for v in t],
                              x_ticks=axis, y_title="mid", table_x="time (UTC)")
vol = (tape.with_columns((pl.col("ts_ns") // step * step).alias("bucket"))
       .group_by("bucket").agg(pl.col("qty").filter(pl.col("buy")).sum().alias("buy"),
                               pl.col("qty").filter(~pl.col("buy")).sum().alias("sell")).sort("bucket"))
b = vol["bucket"].to_numpy()
vol_chart = report.line_chart(b, [("Buy aggressor", vol["buy"].to_numpy()), ("Sell aggressor", vol["sell"].to_numpy())],
                              x_labels=[report._utc(int(v)) for v in b], x_ticks=report._time_axis(int(b[0]), int(b[-1])),
                              y_title="volume", digits=4, table_x="bucket (UTC)")
display(report.frame(report.card("Mid (USDT)", mid_chart) + report.card(f"Traded volume per {every} (BTC)", vol_chart), 960))

In [ ]:
tape.select(pl.col("qty").quantile(q).alias(f"p{int(q * 100)}") for q in (0.1, 0.25, 0.5, 0.75, 0.9, 0.99)).with_columns(
    pl.lit(tape["qty"].mean()).alias("mean"), pl.lit(tape.height).alias("trades"))

## Does the imbalance predict the next move?

At each change of the best prices: the imbalance I = (bid qty - ask qty) / (bid qty + ask qty) at the top
level, and the mid's move over the next second, in ticks. Grouped into five equal-sized bins of I. A rising
pattern means a book leaning to the bid tends to be followed by a rising mid: the signal the imbalance taker
(`research/strategies/imbalance_taker.py`) trades. Whether it beats a tick of spread plus the taker fee is another
question (the strategy walkthrough).

In [ ]:
HORIZON_NS = 1_000_000_000
imb = ((book["bid_qty"] - book["ask_qty"]) / (book["bid_qty"] + book["ask_qty"])).to_numpy()
mid = book["mid"].to_numpy()
later = mid[np.clip(np.searchsorted(t, t + HORIZON_NS, side="right") - 1, 0, len(t) - 1)]
ok = t + HORIZON_NS <= t[-1]
move = (later - mid)[ok] / tick
edges = np.quantile(imb[ok], np.linspace(0, 1, 6))
k = np.clip(np.searchsorted(edges, imb[ok], side="right") - 1, 0, 4)
means = [float(move[k == i].mean()) if (k == i).any() else float("nan") for i in range(5)]
labels = [f"{edges[i]:+.2f} to {edges[i + 1]:+.2f}" for i in range(5)]
display(report.frame(report.card("Mean mid move over the next second (ticks) by imbalance bin", report.bar_chart(
    labels, means, y_title="ticks", digits=3, x_title="imbalance at the top level",
    extra=[("observations", [int((k == i).sum()) for i in range(5)])])), 420))

## What to look for

- On BTCUSDT the spread is one tick nearly all the time, so passive quotes join long queues (the queue
  analytics, `research/analytics/queue.py`).
- Volume comes in bursts; the market's VWAP and the mid's TWAP (`tca.slippage`) part ways in them.
- If the bins rise from left to right, the top-of-book imbalance leads the next second's move. Even then a move
  of a fraction of a tick is less than the cost of crossing the spread plus the taker fee, which is why the
  imbalance taker loses on the recorded hour (docs/strategies.md). One minute of data has few observations
  per bin: run it on the recorded day.